# Testing New Mutation Functions

This notebook tests the newly implemented mutation types:
- `boolean_literal`: True ↔ not False
- `commutative_reorder`: a + b ↔ b + a, a * b ↔ b * a  
- `constant_unfold`: 10 ↔ 5 + 5, 6 ↔ 2 * 3
- `literal_format`: 'hello' ↔ "hello" (may not show changes due to AST normalization)

In [ ]:
import os
import sys
import ast
import random

# Add project root to path
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

print(f"Current directory: {curr_dir}")
print(f"Project directory: {proj_dir}")

In [ ]:
# Import required modules
from code_mutation.mutation_functions import CodeMutator
from code_mutation.ast_mutation import ASTNodeHelper
from database import MongoDBHelper
from code_inconsistency.code_inconsistency_tester import CodeInconsistencyHumanEvalHelper

print("All imports successful!")
print(f"Available mutation types: {CodeMutator.mutation_types}")

## Test 1: Direct AST Transformer Testing

Test each transformer individually to see the mutations in action.

In [ ]:
# Test new mutation types individually (including the new constant unfold variants)
new_mutations = ["boolean_literal", "commutative_reorder", "constant_unfold", "constant_unfold_add", "constant_unfold_mult", "literal_format"]

test_codes = {
    "boolean_literal": """def check_flag():
    flag = True
    other_flag = False
    if flag and not other_flag:
        return "active"
    return "inactive"
""",
    
    "commutative_reorder": """def calculate_sum(a, b, c):
    total = a + b + c
    product = a * b * c  
    mixed = a + (b * c)
    return total, product, mixed
""",
    
    "constant_unfold": """def get_limits():
    min_val = 5
    max_val = 10
    threshold = 8
    large_num = 15
    return min_val, max_val, threshold, large_num
""",

    "constant_unfold_add": """def get_limits_add():
    small = 4
    medium = 7  
    large = 12
    return small, medium, large
""",

    "constant_unfold_mult": """def get_limits_mult():
    six = 6
    ten = 10
    fifteen = 15
    prime = 7  # This should not change (prime number)
    return six, ten, fifteen, prime
""",
    
    "literal_format": """def get_message():
    greeting = 'Hello'
    name = "World"
    punctuation = '!'
    return greeting + " " + name + punctuation
"""
}

print("Testing individual AST transformers (including new constant unfold variants):")
print("="*70)

for mutation_type in new_mutations:
    print(f"\n--- Testing {mutation_type.upper()} ---")
    test_code = test_codes[mutation_type]
    
    try:
        tree = ast.parse(test_code)
        
        if mutation_type == "boolean_literal":
            transformer = ASTNodeHelper.BooleanLiteralTransformer()
        elif mutation_type == "commutative_reorder":
            transformer = ASTNodeHelper.CommutativeReorderTransformer()
        elif mutation_type == "constant_unfold":
            transformer = ASTNodeHelper.ConstantUnfoldTransformer()
        elif mutation_type == "constant_unfold_add":
            transformer = ASTNodeHelper.ConstantUnfoldAddTransformer()
        elif mutation_type == "constant_unfold_mult":
            transformer = ASTNodeHelper.ConstantUnfoldMultTransformer()
        elif mutation_type == "literal_format":
            transformer = ASTNodeHelper.LiteralFormatTransformer()
            
        mutated_tree = transformer.visit(tree)
        ast.fix_missing_locations(mutated_tree)
        mutated_code = ast.unparse(mutated_tree)
        
        print("Original:")
        print(test_code)
        print("\nMutated:")  
        print(mutated_code)
        
        # Check if actually changed
        if ast.unparse(ast.parse(test_code)) != mutated_code:
            print("\n✅ Successfully mutated")
        else:
            print(f"\n⚠️ No change detected for {mutation_type}")
            if mutation_type == "constant_unfold_mult":
                print("   (Expected for prime numbers in multiplication-only mode)")
            elif mutation_type == "literal_format":
                print("   (Expected due to AST normalization)")
        
        # Test compilation
        compile(mutated_code, '<mutated>', 'exec')
        print("✅ Mutated code compiles successfully")
            
    except Exception as e:
        print(f"❌ Error: {e}")
    
    print("-" * 70)

## Test 2: Full Mutation Pipeline Testing

Test using the full CodeMutator pipeline with database connection.

In [ ]:
# Connect to database
try:
    db = MongoDBHelper()
    base_qns_db = db.client["Base_Questions_DB"]
    question_database = base_qns_db['HumanEval_Input_Output']
    print(f"✅ Connected to database. Total documents: {question_database.count_documents({})}")
except Exception as e:
    print(f"⚠️ Database connection failed: {e}")
    print("Will proceed with synthetic examples instead")
    question_database = None

In [ ]:
# Test with synthetic examples (works without database)
synthetic_examples = {
    "boolean_literal": {
        "full_sol": """def is_valid(flag):
    if flag == True:
        return True
    return False
""",
        "examples": {"assert is_valid(True) == True": "True"},
        "qn_desc": "Check if flag is valid",
        "input_args": True,
        "output_args": True
    },
    
    "commutative_reorder": {
        "full_sol": """def add_three(a, b, c):
    return a + b + c
""",
        "examples": {"assert add_three(1, 2, 3) == 6": "6"},
        "qn_desc": "Add three numbers",
        "input_args": [1, 2, 3],
        "output_args": 6
    },
    
    "constant_unfold": {
        "full_sol": """def get_magic_number():
    return 42
""",
        "examples": {"assert get_magic_number() == 42": "42"},
        "qn_desc": "Return the magic number",
        "input_args": [],
        "output_args": 42
    }
}

print("Testing full mutation pipeline with synthetic examples:")
print("="*60)

for mutation_type, example_data in synthetic_examples.items():
    print(f"\n--- Testing {mutation_type.upper()} Pipeline ---")
    
    try:
        mutated_dict = CodeMutator.mutate_for_code_inconsistency_test(
            mutation_type=mutation_type,
            full_sol=example_data["full_sol"],
            examples=example_data["examples"],
            qn_desc=example_data["qn_desc"],
            input_args=example_data["input_args"],
            output_args=example_data["output_args"]
        )
        
        print("✅ Mutation pipeline completed successfully")
        print("Original:")
        print(example_data["full_sol"])
        print("\nMutated:")
        print(mutated_dict['full_sol'])
        
        # Check if changed
        if CodeMutator.standardize_program(example_data["full_sol"]) != CodeMutator.standardize_program(mutated_dict['full_sol']):
            print("\n✅ Code was successfully mutated")
        else:
            print("\n⚠️ No change detected in standardized program")
        
    except Exception as e:
        print(f"❌ Pipeline error: {type(e).__name__}: {e}")
    
    print("-" * 60)

## Test 3: Database Testing (if available)

Test with real database entries if database connection is available.

In [ ]:
if question_database is not None:
    print("Testing with real database entries:")
    print("="*60)
    
    # Find a sample document
    sample_doc = question_database.find_one({"_id": "HumanEvalTF0"})
    
    if sample_doc:
        print(f"Testing with document: {sample_doc['_id']}")
        
        for mutation_type in ["boolean_literal", "commutative_reorder", "constant_unfold"]:
            print(f"\n--- Testing {mutation_type.upper()} with DB ---")
            
            try:
                mutated_dict = CodeMutator.mutate_for_code_inconsistency_test(
                    mutation_type=mutation_type,
                    full_sol=sample_doc['full_sol'],
                    examples=sample_doc['examples'],
                    qn_desc=sample_doc['qn_desc'],
                    input_args=sample_doc['input']['args'],
                    output_args=sample_doc['output']['args']
                )
                
                print("✅ Database mutation successful")
                print("Original:")
                print(sample_doc['full_sol'][:200] + "..." if len(sample_doc['full_sol']) > 200 else sample_doc['full_sol'])
                print("\nMutated:")
                mutated_code = mutated_dict['full_sol']
                print(mutated_code[:200] + "..." if len(mutated_code) > 200 else mutated_code)
                
            except Exception as e:
                print(f"⚠️ Expected: {type(e).__name__}: {e}")
                print("(This may be expected if the code doesn't contain the target constructs)")
    else:
        print("No sample document found")
else:
    print("Database not available - skipping database tests")

## Summary

This notebook tests the new mutation types with improved constant unfolding:

### Core Mutations:
1. **`boolean_literal`**: Transforms `True ↔ not False` and `False ↔ not True`
2. **`commutative_reorder`**: Reorders commutative operations like `a + b ↔ b + a`
3. **`literal_format`**: Attempts `'hello' ↔ "hello"` (limited by AST normalization)

### Constant Unfolding (Enhanced):
4. **`constant_unfold`**: Random choice with fallback - `10 ↔ 5 + 5` OR `2 * 5` (falls back to addition if multiplication fails)
5. **`constant_unfold_add`**: Addition only - `10 ↔ 5 + 5`, `7 ↔ 3 + 4` 
6. **`constant_unfold_mult`**: Multiplication only - `10 ↔ 2 * 5`, `6 ↔ 2 * 3` (no change for primes like 7)

### Key Improvements:
- **Better control**: Can test specific transformations (add-only vs mult-only)
- **Fallback logic**: Random mode falls back to addition if multiplication fails
- **Code reuse**: Shared logic between transformers
- **Predictable behavior**: Addition always works, multiplication only when factorizable

The mutations are now available for use in the consistency testing framework with the mutation types:
`["boolean_literal", "commutative_reorder", "constant_unfold", "constant_unfold_add", "constant_unfold_mult", "literal_format"]`